# Digifly Phase 2 Workbench

Notebook-first public control surface for Phase 2 simulations.

The workbench keeps simulation logic in Python modules and keeps the notebook thin. This provides an app-like Jupyter surface while preserving reusable, testable framework code.

Primary workflows:

- launch shared Phase 2 runs (`single`, `custom`, `hemilineage`)
- launch the fuller hemilineage project pipeline
- preview and validate run plans before execution
- save workbench bundles and manifests for reproducibility
- keep error and fix logs next to the run

Recommended workflow:

1. Set the Phase 1 `export_swc` tree.
2. Select the closest preset.
3. Preview the plan.
4. Validate.
5. Write the bundle.
6. Run.


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path


def _safe_cwd() -> Path:
    try:
        return Path(os.getcwd())
    except FileNotFoundError:
        home = os.environ.get('HOME')
        if home:
            return Path(home)
        return Path('/tmp')


def _find_phase2_root() -> Path:
    candidates = []

    env_root = os.environ.get('DIGIFLY_PHASE2_ROOT', '').strip()
    if env_root:
        candidates.append(Path(env_root))

    cwd = _safe_cwd().expanduser().resolve()
    candidates.append(cwd)
    candidates.extend(cwd.parents)

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.expanduser().resolve()
        except Exception:
            continue
        if str(candidate) in seen:
            continue
        seen.add(str(candidate))
        if (candidate / 'digifly').exists():
            return candidate
    raise RuntimeError('Could not find the Phase 2 root. Set DIGIFLY_PHASE2_ROOT to the folder that contains digifly/.')


PHASE2_ROOT = _find_phase2_root()
REPO_ROOT = PHASE2_ROOT.parent
DEFAULT_SWC_DIR = REPO_ROOT / 'Phase 1' / 'manc_v1.2.1' / 'export_swc'

os.environ.setdefault('DIGIFLY_PHASE2_ROOT', str(PHASE2_ROOT))
os.environ.setdefault('DIGIFLY_WORKSPACE', str(REPO_ROOT))
if DEFAULT_SWC_DIR.exists():
    os.environ.setdefault('DIGIFLY_SWC_DIR', str(DEFAULT_SWC_DIR))

if str(PHASE2_ROOT) not in sys.path:
    sys.path.insert(0, str(PHASE2_ROOT))

# Jupyter kernels cache imports. Clear the workbench package so notebook edits
# and local source changes are picked up when this setup cell is rerun.
for module_name in list(sys.modules):
    if module_name == 'digifly.phase2.workbench' or module_name.startswith('digifly.phase2.workbench.'):
        del sys.modules[module_name]

print(f'Phase 2 root: {PHASE2_ROOT}')
print(f'Default SWC root: {os.environ.get("DIGIFLY_SWC_DIR", "") or "not found"}')

from digifly.phase2.workbench import PRESETS, launch_workbench

print('Available presets:')
for preset in PRESETS:
    print(f'  - {preset.slug}: {preset.label}')


## Workbench Shape

The workbench currently exposes two execution surfaces:

- `shared_runner`: the current `run_walking_simulation()` path for `single`, `custom`, and `hemilineage`
- `hemilineage_project`: the fuller `run_full_hemilineage_project()` path for baseline and pipeline-style Phase 2 work

The quick controls cover common parameters. Advanced JSON boxes expose deeper configuration without turning the notebook into a wall of code.

Artifacts are written in two places:

- a workbench preflight bundle under `Phase 2/workbench_runs/<run_id>/`
- the actual simulation output folder written by the selected runner, usually `SWC_DIR/hemi_runs/<run_id>/` for shared runs

Both locations receive manifest-style files. The workbench also creates `error_log.md` and `fix_log.md` placeholders so each run has a diagnostics trail.


In [ ]:
ui = launch_workbench(PHASE2_ROOT)
ui


In [ ]:
# Optional helper: inspect the latest workbench bundle path.
from pathlib import Path

workbench_runs = (PHASE2_ROOT / 'workbench_runs').resolve()
workbench_runs.mkdir(parents=True, exist_ok=True)
latest = sorted(workbench_runs.iterdir(), key=lambda p: p.stat().st_mtime, reverse=True)
latest[0] if latest else 'No workbench bundles yet.'
